# Etherpad Pad 内容重建查看器

本工具用于：
1. 查询数据库获取所有 room-id 和版本范围
2. SSH 连接到服务器执行重建脚本
3. 获取并展示重建的内容
4. 支持编辑和对比

---

## 📋 功能清单

- ✅ 查询 MySQL 数据库获取所有 Pad 列表
- ✅ 显示每个 Pad 的版本范围
- ✅ SSH 连接到远程服务器执行重建脚本
- ✅ 交互式选择 Pad 和版本范围
- ✅ 查看特定版本的详细内容
- ✅ 对比两个版本的差异
- ✅ 导出结果为 JSON 和 CSV 格式
- ✅ 批量处理多个 Pad


## 1. 安装依赖包

首次使用需要安装必要的 Python 包


In [5]:
# 安装必要的 Python 包
#!pip install pymysql paramiko pandas ipywidgets -q



[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. 导入库和配置


In [14]:
import pymysql
import paramiko
import json
import pandas as pd
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 数据库配置
DB_CONFIG = {
    'host': '112.74.92.135',
    'user': 'root',
    'password': '1q2w3e4R',
    'database': 'alic',
    'charset': 'utf8mb4',
    'port': 3306
}

# SSH 配置
SSH_CONFIG = {
    'host': '8.138.89.124',
    'port': 22,
    'username': 'root',
    'password': 'Alic@root'
}

# 脚本路径
SCRIPT_PATH = '/opt/etherpad-test/src/node/scheduler/etherpad_changes/PadContentRebuildAPI.js'
WORK_DIR = '/opt/etherpad-test/src'

print("✅ 配置加载完成")


✅ 配置加载完成


## 3. 定义数据库查询函数


In [15]:
def get_all_room_ids_with_versions():
    """从 store 表查询所有 room-id 及其版本范围"""
    try:
        connection = pymysql.connect(**DB_CONFIG)
        cursor = connection.cursor(pymysql.cursors.DictCursor)
        
        query = """
        SELECT 
            SUBSTRING_INDEX(SUBSTRING_INDEX(`key`, ':', 2), ':', -1) as pad_id,
            MIN(CAST(SUBSTRING_INDEX(`key`, ':', -1) AS UNSIGNED)) as min_revision,
            MAX(CAST(SUBSTRING_INDEX(`key`, ':', -1) AS UNSIGNED)) as max_revision,
            COUNT(*) as revision_count
        FROM store
        WHERE `key` LIKE 'pad:%:revs:%'
        GROUP BY pad_id
        ORDER BY pad_id
        """
        
        cursor.execute(query)
        results = cursor.fetchall()
        
        cursor.close()
        connection.close()
        
        return results
        
    except Exception as e:
        print(f"❌ 数据库查询失败: {e}")
        return []

print("✅ 数据库查询函数定义完成")


✅ 数据库查询函数定义完成


## 4. 定义 SSH 远程执行函数


In [16]:
def execute_rebuild_script(pad_id, start_rev=None, end_rev=None):
    """SSH 连接到服务器执行重建脚本"""
    try:
        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        
        print(f"🔗 连接到服务器 {SSH_CONFIG['host']}...")
        ssh.connect(
            hostname=SSH_CONFIG['host'],
            port=SSH_CONFIG['port'],
            username=SSH_CONFIG['username'],
            password=SSH_CONFIG['password'],
            timeout=30
        )
        print("✅ SSH 连接成功")
        
        # 构建命令
        cmd_parts = [
            f"cd {WORK_DIR}",
            "&&",
            "node --require tsx/cjs",
            SCRIPT_PATH,
            pad_id
        ]
        
        if start_rev is not None:
            cmd_parts.append(str(start_rev))
            if end_rev is not None:
                cmd_parts.append(str(end_rev))
        
        command = ' '.join(cmd_parts)
        print(f"🚀 执行命令: {command}")
        
        # 执行命令
        stdin, stdout, stderr = ssh.exec_command(command, timeout=300)
        
        # 读取输出
        output = stdout.read().decode('utf-8')
        error = stderr.read().decode('utf-8')
        exit_code = stdout.channel.recv_exit_status()
        
        ssh.close()
        
        if exit_code == 0:
            print("✅ 脚本执行成功")
            try:
                result = json.loads(output)
                return result
            except json.JSONDecodeError as e:
                print(f"❌ JSON 解析失败: {e}")
                return {'success': False, 'error': 'JSON parse error', 'raw_output': output[:500]}
        else:
            print(f"❌ 脚本执行失败 (退出码: {exit_code})")
            print(f"错误信息: {error}")
            return {'success': False, 'error': error, 'exit_code': exit_code}
            
    except Exception as e:
        print(f"❌ SSH 执行失败: {e}")
        return {'success': False, 'error': str(e)}

print("✅ SSH 执行函数定义完成")


✅ SSH 执行函数定义完成


## 5. 查询所有 Room ID 和版本范围


In [17]:
# 查询所有 room-id 及版本范围
print("🔍 正在查询数据库...")
room_list = get_all_room_ids_with_versions()

if room_list:
    df = pd.DataFrame(room_list)
    df.index = df.index + 1
    
    print(f"\n✅ 找到 {len(room_list)} 个 Pad\n")
    display(df)
    
    # 统计信息
    total_revisions = df['revision_count'].sum()
    avg_revisions = df['revision_count'].mean()
    
    print(f"\n📊 统计信息:")
    print(f"   总 Pad 数: {len(room_list)}")
    print(f"   总版本数: {total_revisions}")
    print(f"   平均版本数: {avg_revisions:.2f}")
    print(f"   最多版本: {df['revision_count'].max()} (Pad: {df.loc[df['revision_count'].idxmax(), 'pad_id']})")
    print(f"   最少版本: {df['revision_count'].min()} (Pad: {df.loc[df['revision_count'].idxmin(), 'pad_id']})")
else:
    print("❌ 未找到任何 Pad 数据")
    df = pd.DataFrame()


🔍 正在查询数据库...

✅ 找到 3 个 Pad



,pad_id,min_revision,max_revision,revision_count
1,room-210,0,0,1
2,room-229,0,135,136
3,room-243,0,26,27



📊 统计信息:
   总 Pad 数: 3
   总版本数: 164
   平均版本数: 54.67
   最多版本: 136 (Pad: room-229)
   最少版本: 1 (Pad: room-210)


## 6. 执行 Pad 内容重建

💡 **使用说明：** 修改下面代码中的参数，然后运行单元格

- `pad_id`: 从上面表格中选择一个 Pad ID
- `start_rev`: 起始版本号
- `end_rev`: 结束版本号


In [19]:
# ========================================
# 手动设置参数并执行重建
# ========================================

# 📝 在这里修改参数
pad_id = 'room-229'        # 修改为你要查询的 Pad ID
start_rev = 0              # 起始版本号
end_rev = 10               # 结束版本号

# 执行重建
if not df.empty:
    print(f"\n{'='*80}")
    print(f"📝 Pad ID: {pad_id}")
    print(f"📊 版本范围: {start_rev} - {end_rev}")
    print(f"⏰ 开始时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}\n")
    
    # 执行重建
    rebuild_result = execute_rebuild_script(pad_id, start_rev, end_rev)
    
    if rebuild_result.get('success'):
        stats = rebuild_result.get('statistics', {})
        print(f"\n{'='*80}")
        print(f"✅ 重建完成！")
        print(f"{'='*80}")
        print(f"   总版本数: {stats.get('total', 0)}")
        print(f"   成功: {stats.get('success', 0)}")
        print(f"   失败: {stats.get('failed', 0)}")
        print(f"\n⏰ 完成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"\n💡 结果已保存到变量 rebuild_result，可以在下面的单元格中查看详细内容")
    else:
        print(f"\n{'='*80}")
        print(f"❌ 重建失败")
        print(f"{'='*80}")
        print(f"错误信息: {rebuild_result.get('error', 'Unknown error')}")
else:
    print("❌ 没有可用的 Pad 数据，请先执行上面的查询")



📝 Pad ID: room-229
📊 版本范围: 0 - 10
⏰ 开始时间: 2025-12-10 16:14:16

🔗 连接到服务器 8.138.89.124...
✅ SSH 连接成功
🚀 执行命令: cd /opt/etherpad-test/src && node --require tsx/cjs /opt/etherpad-test/src/node/scheduler/etherpad_changes/PadContentRebuildAPI.js room-229 0 10
❌ 脚本执行失败 (退出码: 1)
错误信息: /opt/etherpad-test/src/node/scheduler/etherpad_changes/PadContentRebuildAPI.js:15
process.on('unhandledRejection', (err) => { throw err; });
                                            ^

TypeError: db.close is not a function
    at rebuildPadContent (/opt/etherpad-test/src/node/scheduler/etherpad_changes/PadContentRebuildAPI.js:186:14)
    at process.processTicksAndRejections (node:internal/process/task_queues:95:5)
    at async /opt/etherpad-test/src/node/scheduler/etherpad_changes/PadContentRebuildAPI.js:192:18

Node.js v18.20.8


❌ 重建失败
错误信息: /opt/etherpad-test/src/node/scheduler/etherpad_changes/PadContentRebuildAPI.js:15
process.on('unhandledRejection', (err) => { throw err; });
                             

## 7. 查看完整的重建结果（JSON 格式）


In [ ]:
# 查看完整的 JSON 结果
if 'rebuild_result' in globals() and rebuild_result:
    print("📄 完整 JSON 结果:")
    print(json.dumps(rebuild_result, indent=2, ensure_ascii=False))
else:
    print("⚠️ 请先在上面的控制面板中执行重建")


## 8. 版本详情查看器 - 查看特定版本的内容


In [ ]:
# 查看特定版本的详细内容
if 'rebuild_result' in globals() and rebuild_result and rebuild_result.get('success'):
    versions = rebuild_result.get('versions', [])
    
    if versions:
        print(f"✅ 共有 {len(versions)} 个版本")
        print(f"\n💡 输入要查看的版本号 (0-{len(versions)-1}):")
        
        # 显示版本列表
        print("\n可用版本列表:")
        print("-" * 80)
        for i, v in enumerate(versions[:20]):  # 只显示前20个
            if v.get('success'):
                print(f"  版本 {v['revision']}: {v.get('formatted_timestamp', 'N/A')}")
        if len(versions) > 20:
            print(f"  ... 还有 {len(versions) - 20} 个版本")
    else:
        print("⚠️ 没有可用的版本数据")
else:
    print("⚠️ 请先在上面执行重建")


## 9. 查看指定版本的详细内容

修改 `view_revision` 参数来查看不同版本


In [ ]:
# 📝 修改这里的版本号
view_revision = 5

# 查看版本详情
if 'rebuild_result' in globals() and rebuild_result and rebuild_result.get('success'):
    versions = rebuild_result.get('versions', [])
    version = next((v for v in versions if v.get('revision') == view_revision), None)
    
    if version and version.get('success'):
        print(f"\n{'='*80}")
        print(f"📝 版本 {view_revision} 详细信息")
        print(f"{'='*80}")
        print(f"Pad ID:        {version.get('pad_id', 'N/A')}")
        print(f"作者:          {version.get('author', 'N/A')}")
        print(f"时间戳:        {version.get('timestamp', 'N/A')}")
        print(f"格式化时间:    {version.get('formatted_timestamp', 'N/A')}")
        print(f"文本长度:      {version.get('text_length', 0)} 字符")
        print(f"行数:          {version.get('line_count', 0)}")
        print(f"变更摘要:      {version.get('change_summary', 'N/A')}")
        
        print(f"\n{'='*80}")
        print(f"📄 文本内容")
        print(f"{'='*80}")
        print(version.get('content', ''))
        
        print(f"\n{'='*80}")
        print(f"🔧 Changeset")
        print(f"{'='*80}")
        print(version.get('changeset', 'N/A'))
        
        print(f"\n{'='*80}")
        print(f"🎨 Attributes")
        print(f"{'='*80}")
        attribs = version.get('attribs', '')
        print(attribs if attribs else '(无属性)')
    else:
        print(f"❌ 未找到版本 {view_revision}")
else:
    print("⚠️ 请先执行重建")


## 10. 版本对比

对比两个版本的差异


## 测试本地 PadContentRebuildAPI.js

使用本地的 API 脚本来获取 pad 版本数据


In [21]:
import subprocess
import json
import os

# 设置工作目录
project_root = r"D:\ALIC\alic-etherpad-lite"
src_dir = os.path.join(project_root, "src")

# 测试参数
test_pad_id = "room-229"  # 可以修改为其他 pad ID
start_rev = 0
end_rev = 10

# 构建命令
api_script = os.path.join(project_root, "src", "node", "scheduler", "etherpad_changes", "PadContentRebuildAPI.js")
cmd = [
    "node",
    "--require", "tsx/cjs",
    api_script,
    test_pad_id,
    str(start_rev),
    str(end_rev)
]

print(f"执行命令: {' '.join(cmd)}")
print(f"工作目录: {src_dir}")
print("-" * 80)

try:
    # 执行命令 - 使用 utf-8 编码避免 Windows GBK 编码问题
    result = subprocess.run(
        cmd,
        cwd=src_dir,
        capture_output=True,
        text=True,
        encoding='utf-8',
        errors='replace',  # 遇到无法解码的字符时替换为 �
        timeout=30
    )
    
    print(f"返回码: {result.returncode}")
    
    # 安全检查输出是否为 None
    stdout_len = len(result.stdout) if result.stdout else 0
    stderr_len = len(result.stderr) if result.stderr else 0
    
    print(f"标准输出长度: {stdout_len} 字符")
    print(f"标准错误长度: {stderr_len} 字符")
    print("-" * 80)
    
    if result.stderr:
        print("标准错误:")
        print(result.stderr)
        print("-" * 80)
    
    if result.stdout:
        try:
            # 解析 JSON 输出
            data = json.loads(result.stdout)
            print("✅ JSON 解析成功!")
            print(f"成功: {data.get('success')}")
            print(f"Pad ID: {data.get('pad_id')}")
            print(f"Head Revision: {data.get('head_revision')}")
            
            if 'statistics' in data:
                stats = data['statistics']
                print(f"统计: 总计={stats.get('total')}, 成功={stats.get('success')}, 失败={stats.get('failed')}")
            
            if 'versions' in data:
                versions = data['versions']
                print(f"版本数量: {len(versions)}")
                
                # 显示前几个版本的摘要
                print("\n前3个版本:")
                for i, ver in enumerate(versions[:3]):
                    if ver.get('success'):
                        print(f"  Rev {ver['revision']}: {ver.get('text_length')} 字符, {ver.get('line_count')} 行")
                        print(f"    时间: {ver.get('formatted_timestamp')}")
                        print(f"    作者: {ver.get('author')}")
                        if ver.get('content'):
                            preview = ver['content'][:100].replace('\n', '\\n')
                            print(f"    内容预览: {preview}...")
                    else:
                        print(f"  Rev {ver['revision']}: ❌ {ver.get('error')}")
            
            # 保存完整数据供后续分析
            api_test_result = data
            print("\n✅ 数据已保存到变量 'api_test_result'")
            
        except json.JSONDecodeError as e:
            print(f"❌ JSON 解析失败: {e}")
            print("原始输出:")
            print(result.stdout[:1000])
    else:
        print("⚠️ 没有标准输出")
        
except subprocess.TimeoutExpired:
    print("❌ 命令执行超时 (30秒)")
except Exception as e:
    print(f"❌ 执行出错: {e}")
    import traceback
    traceback.print_exc()

api_test_result


执行命令: node --require tsx/cjs D:\ALIC\alic-etherpad-lite\src\node\scheduler\etherpad_changes\PadContentRebuildAPI.js room-229 0 10
工作目录: D:\ALIC\alic-etherpad-lite\src
--------------------------------------------------------------------------------
返回码: 0
标准输出长度: 5448 字符
标准错误长度: 0 字符
--------------------------------------------------------------------------------
✅ JSON 解析成功!
成功: True
Pad ID: room-229
Head Revision: 135
统计: 总计=11, 成功=11, 失败=0
版本数量: 11

前3个版本:
  Rev 0: 228 字符, 6 行
    时间: 2025-09-17 23:39:12.431
    作者: a.ni6xvsCFoJs9Rr1v
    内容预览: Welcome to Etherpad!\n\nThis pad text is synchronized as you type, so that everyone viewing this page ...
  Rev 1: 22 字符, 3 行
    时间: 2025-09-20 22:15:52.139
    作者: a.ni6xvsCFoJs9Rr1v
    内容预览: Welcome to Etherpad!\n\n...
  Rev 2: 24 字符, 3 行
    时间: 2025-09-20 22:15:58.322
    作者: a.ni6xvsCFoJs9Rr1v
    内容预览: Welcome to Etherpad!\n\n周末...

✅ 数据已保存到变量 'api_test_result'


{'success': True,
 'pad_id': 'room-229',
 'head_revision': 135,
 'requested_range': {'start': 0, 'end': 10},
 'statistics': {'total': 11, 'success': 11, 'failed': 0},
 'versions': [{'revision': 0,
   'success': True,
   'pad_id': 'room-229',
   'content': 'Welcome to Etherpad!\n\nThis pad text is synchronized as you type, so that everyone viewing this page sees the same text. This allows you to collaborate seamlessly on documents!\n\nGet involved with Etherpad at https://etherpad.org\n',
   'author': 'a.ni6xvsCFoJs9Rr1v',
   'timestamp': 1758123552431,
   'formatted_timestamp': '2025-09-17 23:39:12.431',
   'changeset': 'Z:1>6c|5+6c$Welcome to Etherpad!\n\nThis pad text is synchronized as you type, so that everyone viewing this page sees the same text. This allows you to collaborate seamlessly on documents!\n\nGet involved with Etherpad at https://etherpad.org\n',
   'attribs': '|6+6d',
   'text_length': 228,
   'line_count': 6,
   'change_summary': '1 -> 229 chars'},
  {'revision': 1,

In [ ]:
# 📝 修改这里的版本号
compare_rev1 = 5
compare_rev2 = 6

# 对比两个版本
if 'rebuild_result' in globals() and rebuild_result and rebuild_result.get('success'):
    versions = rebuild_result.get('versions', [])
    v1 = next((v for v in versions if v.get('revision') == compare_rev1), None)
    v2 = next((v for v in versions if v.get('revision') == compare_rev2), None)
    
    if v1 and v2 and v1.get('success') and v2.get('success'):
        print(f"\n{'='*80}")
        print(f"🔄 版本对比: {compare_rev1} vs {compare_rev2}")
        print(f"{'='*80}\n")
        
        # 基本信息对比
        print(f"{'属性':<20} {'版本 ' + str(compare_rev1):<35} {'版本 ' + str(compare_rev2):<35}")
        print("-" * 90)
        print(f"{'文本长度':<20} {v1.get('text_length', 0):<35} {v2.get('text_length', 0):<35}")
        print(f"{'行数':<20} {v1.get('line_count', 0):<35} {v2.get('line_count', 0):<35}")
        print(f"{'作者':<20} {v1.get('author', 'N/A'):<35} {v2.get('author', 'N/A'):<35}")
        print(f"{'时间':<20} {v1.get('formatted_timestamp', 'N/A')[:19]:<35} {v2.get('formatted_timestamp', 'N/A')[:19]:<35}")
        print(f"{'变更摘要':<20} {v1.get('change_summary', 'N/A'):<35} {v2.get('change_summary', 'N/A'):<35}")
        
        # 内容对比
        content1 = v1.get('content', '')
        content2 = v2.get('content', '')
        
        print(f"\n{'='*80}")
        print(f"📄 内容对比")
        print(f"{'='*80}\n")
        
        if content1 == content2:
            print("✅ 两个版本内容完全相同")
        else:
            print(f"⚠️ 两个版本内容不同")
            print(f"\n长度变化: {len(content1)} → {len(content2)} ({len(content2) - len(content1):+d} 字符)")
            
            lines1 = content1.split('\n')
            lines2 = content2.split('\n')
            print(f"行数变化: {len(lines1)} → {len(lines2)} ({len(lines2) - len(lines1):+d} 行)")
            
            print(f"\n{'='*80}")
            print(f"版本 {compare_rev1} 内容预览:")
            print(f"{'='*80}")
            print(content1[:300] + ('...' if len(content1) > 300 else ''))
            
            print(f"\n{'='*80}")
            print(f"版本 {compare_rev2} 内容预览:")
            print(f"{'='*80}")
            print(content2[:300] + ('...' if len(content2) > 300 else ''))
    else:
        print(f"❌ 未找到版本 {compare_rev1} 或 {compare_rev2}")
else:
    print("⚠️ 请先执行重建")


## 11. 导出结果

将重建结果导出为 JSON 和 CSV 文件


In [ ]:
# 导出结果到文件
if 'rebuild_result' in globals() and rebuild_result and rebuild_result.get('success'):
    pad_id = rebuild_result.get('pad_id', 'unknown')
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # 导出 JSON
    json_filename = f"rebuild_{pad_id}_{timestamp}.json"
    with open(json_filename, 'w', encoding='utf-8') as f:
        json.dump(rebuild_result, f, indent=2, ensure_ascii=False)
    print(f"✅ JSON 结果已保存到: {json_filename}")
    
    # 导出 CSV (版本列表)
    versions = rebuild_result.get('versions', [])
    if versions:
        versions_data = []
        for v in versions:
            if v.get('success'):
                versions_data.append({
                    'revision': v.get('revision'),
                    'author': v.get('author'),
                    'timestamp': v.get('formatted_timestamp'),
                    'text_length': v.get('text_length'),
                    'line_count': v.get('line_count'),
                    'change_summary': v.get('change_summary'),
                    'content_preview': v.get('content', '')[:100].replace('\n', ' ')
                })
        
        versions_df = pd.DataFrame(versions_data)
        csv_filename = f"rebuild_{pad_id}_{timestamp}.csv"
        versions_df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        print(f"✅ CSV 结果已保存到: {csv_filename}")
        
        print(f"\n📊 版本列表预览:")
        display(versions_df.head(10))
else:
    print("⚠️ 请先执行重建")


## 12. 使用说明

### 完整使用流程

1. **安装依赖** - 运行第1个单元格
2. **导入配置** - 运行第2个单元格
3. **定义函数** - 运行第3-4个单元格
4. **查询 Pad 列表** - 运行第5个单元格，查看所有可用的 Pad
5. **执行重建** - 修改第6个单元格的参数，然后运行
6. **查看结果** - 运行后续单元格查看详细内容

### 参数说明

```python
# 第6个单元格 - 执行重建
pad_id = 'room-229'    # 从查询结果中选择 Pad ID
start_rev = 0          # 起始版本号（通常从0开始）
end_rev = 10           # 结束版本号

# 第9个单元格 - 查看特定版本
view_revision = 5      # 要查看的版本号

# 第10个单元格 - 版本对比
compare_rev1 = 5       # 对比版本1
compare_rev2 = 6       # 对比版本2
```

### 示例工作流

```python
# 1. 找到你感兴趣的 Pad
# 运行第5个单元格后，从输出表格中找到 pad_id

# 2. 重建内容
pad_id = 'room-229'
start_rev = 0
end_rev = 50
# 运行第6个单元格

# 3. 查看特定版本
view_revision = 10
# 运行第9个单元格

# 4. 对比版本
compare_rev1 = 10
compare_rev2 = 20
# 运行第10个单元格

# 5. 导出结果
# 运行第11个单元格
```
